##01. Mount your Google Drive

In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set working directory and clone repository dynamically

import os, sys, shutil, json

# 1. Establish persistent folder on Google Drive for caching
drive_hvac_dir = "/content/drive/MyDrive/SmartBEM-Studio"
os.makedirs(drive_hvac_dir, exist_ok=True)

# 2. Establish local repository runtime folder
local_repo_path = "/content/SmartBEM-Studio"
%cd /content
if os.path.exists(local_repo_path):
    shutil.rmtree(local_repo_path)

print("Cloning repository from GitHub...")
!git clone https://github.com/kethshen/SmartBEM-Studio.git {local_repo_path}

colab_dir = os.path.join(local_repo_path, "backend_server")

# 3. Ensure eplus utility is copied for imports
eplus_src = os.path.join(local_repo_path, "EnergyPlus utility")
eplus_dest = os.path.join(colab_dir, "eplus")
if not os.path.exists(eplus_dest):
    shutil.copytree(eplus_src, eplus_dest)

%cd {colab_dir}
sys.path.append(colab_dir)
print(f"Working directory set to: {os.getcwd()}")

# 4. Load Ngrok Authtoken from Google Colab Secrets
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')

    # Save it to local secrets.json so fastapi_server.py can parse it
    local_secrets = os.path.join(colab_dir, "secrets.json")
    with open(local_secrets, "w") as f:
        json.dump({"NGROK_AUTHTOKEN": ngrok_token}, f, indent=4)
    print("✅ Successfully loaded NGROK_AUTHTOKEN from Colab Secrets!")
except Exception as e:
    print("⚠️ Colab Secret 'NGROK_AUTHTOKEN' not found or failed to load.")
    print("Please click the key icon (🔑) in the Colab sidebar, add a secret named 'NGROK_AUTHTOKEN', grant access, and rerun.")


## 02. Install OpenStudio SDK


In [ ]:
# Run this in a new cell at the top of your notebook:
from eplus import prepare_colab_eplus
prepare_colab_eplus()


##03. Install Ngrok and FastAPI

In [ ]:
!pip install pyngrok nest-asyncio fastapi uvicorn pydantic


In [ ]:
from core.fastapi_server import start_server
start_server(port=8000)


##04. Install Ollama local server

In [ ]:
# 1. Install missing dependencies & GPU utils
!sudo apt-get install -y zstd
!sudo apt update && sudo apt install pciutils lshw -y

In [ ]:
# 2. Install Ollama Linux Engine
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. Install Python SDK
!pip install ollama

### Optional - Pull Ollama models from Ollama web

In [ ]:
import os
os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/SmartBEM-Studio/ollama_models/'


In [ ]:
!ollama pull gemma3:12b


In [ ]:
!ollama list

### Optional -  To remove a model use below

In [ ]:
import subprocess
# Remove gemma manifest
!rm -rf "'/content/drive/MyDrive/SmartBEM-Studio/ollama_models/manifests/registry.ollama.ai/library/gemma3"


In [ ]:
# Easiest: just run ollama rm while OLLAMA_MODELS points to Drive
import os
os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/SmartBEM-Studio/ollama_models"
!ollama rm gemma3:4b


##05. Copy Ollama model from Drive to Colab

In [ ]:
# 2. Safely copy the massive brain from Drive to Colab's fast internal disk (~90 seconds)
import os
import shutil

drive_models_dir = "/content/drive/MyDrive/SmartBEM-Studio/ollama_models"
local_models_dir = "/root/.ollama/models"

if os.path.exists(drive_models_dir) and os.listdir(drive_models_dir):
    print("Copying model from Drive to local disk...")
    os.makedirs(local_models_dir, exist_ok=True)
    for item in os.listdir(drive_models_dir):
        s = os.path.join(drive_models_dir, item)
        d = os.path.join(local_models_dir, item)
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d)
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
    print("Copy complete!")
else:
    print("No cached models found on Google Drive. Will download on-demand.")


##06. Run Ollama local server

In [ ]:
# 4. Set host environment variables to allow connections
import os
import time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

In [ ]:
# 3. Boot Server in a directory that will never be deleted to avoid getcwd errors on rerun
# First, kill any existing stale ollama instances
!pkill -f ollama
time.sleep(1)

# Move to /content to start the background process safely
%cd /content
!nohup ollama serve > /content/ollama_serve.log 2>&1 &
time.sleep(5)

# Return to the active workspace directory
%cd /content/SmartBEM-Studio/backend_server
print("✅ Ollama Server Awake and ready!")


In [ ]:
# 6. Test if it is running
!curl http://localhost:11434

## 07. Test Ollama server and model up and runnig

In [ ]:
"""### Testing on Text Input"""

import ollama
response = ollama.chat(model='gemma3:12b', messages=[
  {'role': 'user', 'content': 'what LLM model are you, answer short'},
])
print(response['message']['content'])

##08. Install and run EnergyPlus

In [ ]:
# 1. Install Backend Dependencies
!pip install -r requirements.txt

# 2. Install Advisor's EnergyPlus Utility (from GitHub)
# Note: This installs the specific dev branch required for EKF hooks
!pip install -q "energy-plus-utility @ git+https://github.com/mugalan/energy-plus-utility.git@dev"

print("Dependencies installed.")

In [ ]:
# Ensure the directory and log file exist, then tail
!mkdir -p /tmp/smartbem_sim_runs && touch /tmp/smartbem_sim_runs/backend.log



In [ ]:
!tail -n 100 -f /tmp/smartbem_sim_runs/backend.log
